# Lesson 9: Leading eval culture (the lead engineer's playbook)

Companion to `curriculum/09-leading-eval-culture.md`.

**Big idea:** the hard part isn't the code. It's getting a team to agree on what "good" means and to measure
it habitually. That's leadership work.

This notebook is mostly **you writing**: it turns the course into a plan for the validator.

## 1. The maturity ladder

| Level | Looks like |
|---|---|
| 0. Vibes | Manual spot-checks; "looks good to me" |
| 1. Golden set | A shared dataset and a script anyone can run |
| 2. Eval-gated changes | Every AI PR includes an eval comparison; guardrails block merges |
| 3. Production loop | Online monitoring; failures flow back into the dataset; quality SLOs |
| 4. Eval-driven org | Requirements arrive with success criteria; decisions are made from evals |

**Aim one rung up, not five.**

In [ ]:
# ✍️ Where is your team today (0-4), and what's the ONE thing that gets you to the next rung?
current_level = None
next_step = ""
print(f"today: level {current_level}  ->  next: {next_step or '(fill me in)'}")

## 2. Communicating upward: translate to business terms

"94% agreement" means little to a director. "About 12 mis-rated Issues a month, 0–3 of them under-rated
by two levels" does. Fill in realistic numbers for your context:

In [ ]:
from evalkit.stats import wilson

issues_per_month = 400        # ✍️ your volume
eval_n = 200                  # ✍️ golden-set size
eval_correct = 188            # ✍️ how many the validator got right
eval_severe_under = 1         # ✍️ how many were under-rated by 2+ levels

lo, hi = wilson(eval_correct, eval_n)
slo, shi = wilson(eval_severe_under, eval_n)
print(f"accuracy {eval_correct/eval_n:.0%} (95% CI {lo:.0%}-{hi:.0%})")
print(f"=> roughly {issues_per_month*(1-hi):.0f}-{issues_per_month*(1-lo):.0f} mis-rated Issues per month")
print(f"=> roughly {issues_per_month*slo:.0f}-{issues_per_month*shi:.0f} severely under-rated Issues per month")

### 🧠 Check
Notice the *severe* range is wide even with 200 cases, because the event is rare. What does that tell you about
how to prove a "never under-rate by 2 levels" guarantee? (Hint: a larger targeted set of high-severity
Issues, plus production sampling.)

## 3. Common failure modes

| Symptom | Likely cause | Response |
|---|---|---|
| Scores high, users complain | Dataset ≠ production | Sample production, add real failures |
| Scores bounce between runs | Too few cases/trials; uncalibrated judge | More cases, paired comparison, calibrate |
| Nobody runs the eval | Slow, expensive, hard | One command, fast smoke set |
| Endless prompt tweaking | No hypothesis | "Which failure group does this target?" |
| Eval became the target | Goodhart | Holdout, rotate cases, read outputs |
| Arguments over which prompt is better | No agreed criteria | Write criteria *before* the next experiment |

### 🧠 Discussion
Which two of these are most likely on *your* team in the first 90 days, and what would you do pre-emptively?

## 4. ✍️ Your 90-day plan for the validator

Fill this in. It becomes the input to `/apply` (the Capstone), which drafts a full strategy doc
from `playbook/team-rollout-90-days.md`.

**Days 1–30: golden set and baseline**
- Success criteria agreed with: ___
- Golden set: ___ Issues, labelled/confirmed by ___, covering ___
- Baseline result: ___

**Days 31–60: eval-gated changes**
- Every prompt/model change needs: ___
- Guardrails that block a merge: ___
- Validator calibrated against ___ with target kappa ___

**Days 61–90: production loop**
- Signals monitored: ___ (override rate? re-rating by 2nd line?)
- Weekly failure review owner: ___
- Results presented to: ___ as: ___

## 5. Your levers

- **Standards:** an "AI feature definition of done" (see `playbook/ai-feature-pr-checklist.md`).
- **Reviews:** ask "what did the eval say?", kindly and consistently.
- **Infrastructure:** one shared, boring harness beats five bespoke notebooks. (These notebooks are learning aids; your team's eval should be a script in CI.)
- **People:** risk SMEs own "what good means" as much as engineers do. Rotate "eval duty". Celebrate added test cases.

---
**Course recap**
- An eval = dataset → task → grader → metric, made repeatable.
- Your validator *is* a grader, so its quality is its calibrated agreement with expert humans, with error bars.
- The culture (criteria first, gated changes, the production → dataset loop) is the lead engineer's job.

**Next:** `/apply` to draft your team's eval strategy (Capstone).